# 15 — Adaptive Geometry (Qubits-on-the-Go): E-series ledger

**Thesis (user):** neutral atoms can restructure their register per simulation — so geometry becomes learnable, input-conditioned, and continually adapted. Design: 13-adaptive-geometry.md + 14-structure-bakeoff.md. This notebook recomputes every E-series verdict from `runs/ag*_*.log` on re-execution.

In [1]:
import sys, pathlib, glob, re, warnings
warnings.filterwarnings('ignore')
ROOT = pathlib.Path().resolve().parent
sys.path.insert(0, str(ROOT / 'paper'))
import numpy as np, pandas as pd, torch
import matplotlib.pyplot as plt
plt.rcParams['figure.dpi'] = 90
DEVICE = 'cpu'   # GPU reserved for the sprint queue; flip when free
torch.manual_seed(0)


In [2]:
def parse_ag(pattern):
    rows = []
    for f in sorted(glob.glob(str(ROOT / 'runs' / pattern))):
        txt = open(f).read()
        g = lambda k: re.search(rf'^{k}:\s+(\S+)', txt, re.M)
        if not g('val_vrmse_1'):
            continue
        m = re.match(r'ag(\d+)_(.+?)_k(\d+)_s(\d+)', pathlib.Path(f).stem)
        rows.append(dict(wave=m.group(1), variant=m.group(2),
                         k=int(m.group(3)), seed=int(m.group(4)),
                         v1=float(g('val_vrmse_1').group(1)),
                         v8=float(g('val_vrmse_8').group(1)),
                         steps=int(g('num_steps').group(1))))
    return pd.DataFrame(rows)
ag = parse_ag('ag*_*.log')
ag.sort_values(['wave','k','v8']).reset_index(drop=True)

,wave,variant,k,seed,v1,v8,steps
0,1,cg30-3d,1,1337,0.437580,0.887687,3063
1,1,mlp30,1,1337,0.424892,1.007769,5049
2,1,cg30,1,1337,0.437850,1.108293,3105
3,1,mr30ryd,1,1337,0.484923,1.115634,3068
4,1,kmix30,1,1337,0.482629,1.575401,1455
5,1,mlp30,4,1337,0.534349,0.885209,1335
6,1,cg30,4,1337,0.586627,0.912490,779
7,1,none,4,1337,0.546324,0.977484,1326
8,1,mr30ryd,4,1337,0.625937,1.038007,789
9,15,cg30,1,1339,0.453168,0.916497,2000


## Fair pairings (matched steps, matched objective) — the E1/E3 verdicts

In [3]:
w1 = ag[ag.wave == '1']
pairs = [('cg30','mr30ryd',1,'E1: conditioned vs static'),
         ('cg30','mr30ryd',4,'E1 under TD'),
         ('cg30-3d','cg30',1,'E3: 3D vs 2D')]
for a, b, k, label in pairs:
    ra = w1.query('variant==@a and k==@k'); rb = w1.query('variant==@b and k==@k')
    if len(ra) and len(rb):
        va, vb = ra.v8.iloc[0], rb.v8.iloc[0]
        sa, sb = ra.steps.iloc[0], rb.steps.iloc[0]
        fair = abs(sa-sb)/max(sa,sb) < 0.1
        print(f'{label:28s} {a} {va:.3f} vs {b} {vb:.3f} '
              f'({100*(va/vb-1):+.1f}%) steps {sa}/{sb} fair={fair}')

E1: conditioned vs static    cg30 1.108 vs mr30ryd 1.116 (-0.7%) steps 3105/3068 fair=True
E1 under TD                  cg30 0.912 vs mr30ryd 1.038 (-12.1%) steps 779/789 fair=True
E3: 3D vs 2D                 cg30-3d 0.888 vs cg30 1.108 (-19.9%) steps 3063/3105 fair=True


## Wave 1.5 — fixed-step deconfounded grid (populates live)

In [4]:
w15 = ag[ag.wave == '15']
if len(w15) == 0:
    print('wave 1.5 running — re-execute when WAVE15_DONE')
else:
    piv = w15.pivot_table(index='variant', columns=['k','seed'], values='v8')
    print(piv.round(3).to_string())
    if len(w15) >= 12:
        m = w15.groupby(['variant','k']).v8.agg(['mean','std','count'])
        print(); print(m.round(3).to_string())

k            1                    4              
seed      1337   1338   1339   1337   1338   1339
variant                                          
cg30     0.928  1.285  0.916  1.050  0.845  0.790
mlp30    1.000  1.114  1.058  1.098  0.933  0.829
mr30ryd  1.323  0.970  1.008  0.903  0.826  0.806
none     0.969  0.962  0.960  1.246  0.860  0.764

            mean    std  count
variant k                     
cg30    1  1.043  0.209      3
        4  0.895  0.137      3
mlp30   1  1.057  0.057      3
        4  0.953  0.135      3
mr30ryd 1  1.100  0.194      3
        4  0.845  0.051      3
none    1  0.964  0.005      3
        4  0.957  0.255      3


## Standing verdicts (wave-1, seed 1337, honest caveats)
- **E1 supported 2/2 fair pairings**: conditioned > static at k=1 (1.108 vs 1.116) and k=4 (0.912 vs 1.038, −12%).
- **E3 strong**: 3D conditioned 0.888 vs 2D 1.108 (−20%) — best k=1 rollout in program; 3D = simulation-only on Pasqal today.
- **E4 double-edged**: TD objective un-starves the baseline (−13%) AND the classical control (0.885, wave-best wall-clock) — the quantum-specific claim must survive wave-1.5's fixed-step stats.
- **kmix30 pathology**: soft mixture evaluates all banks (4× cost, averaged dynamics) — hard Gumbel routing required before the bake-off uses it.
- Single-seed wave-1 numbers are directional; wave-1.5 (3 seeds, fixed 2,000 steps) is the statistical referee.